In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Lambda
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K

In [3]:
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "a man is walking on the street",
    "a dog is barking at the cat",
    "the street is busy with people"
]

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 21


In [5]:
sequences = tokenizer.texts_to_sequences(corpus)
window_size = 2
data = []

for sequence in sequences:
    for i, word in enumerate(sequence):
        context = []

        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(sequence):
                context.append(sequence[j])

        if context:
            data.append((context, word))
            

In [8]:
X_context = [item[0] for item in data]
y_target = np.array([item[1] for item in data])

max_len = max(len(x) for x in X_context)

X_context_padded = pad_sequences(
    X_context, maxlen=max_len, padding='post'
)

y_target_one_hot = to_categorical(
    y_target, num_classes=vocab_size
)

In [9]:
embedding_dim = 100

input_context = Input(shape=(max_len,))

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=max_len
)(input_context)

lambda_layer = Lambda(
    lambda x: K.mean(x, axis=1),
    output_shape=(embedding_dim,)
)(embedding_layer)

output_layer = Dense(
    vocab_size, activation='softmax'
)(lambda_layer)

cbow_model = Model(
    inputs=input_context,
    outputs=output_layer
)

C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
cbow_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining the CBOW model...")

cbow_model.fit(
    X_context_padded,
    y_target_one_hot,
    epochs=500,
    verbose=0
)

print("Training complete.")


Training the CBOW model...
Training complete.


In [13]:
loss, accuracy = cbow_model.evaluate(
    X_context_padded,
    y_target_one_hot,
    verbose=0
)

print("Loss:", loss)
print("Accuracy:", accuracy*100)

Loss: 0.19544324278831482
Accuracy: 94.73684430122375


In [14]:
word_embeddings = cbow_model.layers[1].get_weights()[0]

print("\nLearned Word Embeddings:")

for word, i in word_index.items():
    if i < vocab_size:
        print(f"Word: '{word}', Vector: {word_embeddings[i][:5]}...")


Learned Word Embeddings:
Word: 'the', Vector: [ 0.30317155 -0.26715186  0.28916854  0.28706822  0.8211589 ]...
Word: 'dog', Vector: [ 0.25192115  0.17357378  0.10259321 -0.17201577 -0.36906302]...
Word: 'is', Vector: [-0.11708705  0.16401641 -0.06932924  0.21945158  0.07612015]...
Word: 'quick', Vector: [-0.35347113  0.5980081   0.39223787  0.39956206 -0.43805635]...
Word: 'brown', Vector: [-0.49011913  0.28079635  0.521118   -0.25187454  0.31620693]...
Word: 'fox', Vector: [ 0.11370408 -0.12782332  0.28975126 -0.29928586 -0.2595212 ]...
Word: 'jumps', Vector: [ 0.08156165  0.70399666  0.4825508   0.3248216  -0.34447306]...
Word: 'over', Vector: [-0.7118422  -0.5525519   0.43194342 -0.45165482  0.04772316]...
Word: 'lazy', Vector: [ 0.01912292  0.5445154   0.40970096  0.0321967  -0.28853095]...
Word: 'a', Vector: [ 0.19915037 -0.11342793 -0.30368605  0.09019248 -0.41366154]...
Word: 'street', Vector: [ 0.20996986 -0.48021522 -0.4204627   0.26483482 -0.48085144]...
Word: 'man', Vector: